# Reset extraction — clear runs so `extract_stage` can re-run
Scope: one `(tenant, project)`. Jev responses, selections and passages are **kept**. Cleared:
- `llm_extraction_run` rows (succeeded and failed) — needed because `extract_stage` reuses any succeeded run with an identical version;
- `validation_failure` rows for stage `extraction`;
- `alias_evidence` rows pointing at an extraction run;
- `passage_processing`: `failed` / `extracted` / `selected` passages that have an active Jev selection go back to `selected` with `active_extraction_run_id` and `validated_extraction_run_id` cleared, which is what `extract_stage` picks up (`structural_only` / `not_run` passages are untouched).

Safety: refuses to run if extractions were already materialized (`entity_mention`, `assertion`, `materialization_run`) — those need the graph rebuilt. The reset runs in one transaction and **rolls back while `DRY_RUN = True`**.

In [ ]:
from pathlib import Path
import json
from sqlalchemy import text
from potency_kg.db import create_engine_from_env

nb_dir = Path.cwd().resolve()
root = nb_dir.parent if nb_dir.name == "notebooks" else nb_dir
scope = json.loads((root / "configs/casgevy-potency.json").read_text())["scope"]
P = {"tenant": scope["tenant_id"], "project": scope["project_id"]}
engine = create_engine_from_env()
print("scope:", P)

## 1. Current state

In [ ]:
SCOPE = "tenant_id=:tenant AND project_id=:project"
def snapshot(c):
    print("== llm_extraction_run by status ==")
    for s, n, raw in c.execute(text(f"SELECT status, count(*), count(raw_response) FROM llm_extraction_run WHERE {SCOPE} GROUP BY 1"), P): print(f"  {s:10s} {n:5d}  (with raw_response: {raw})")
    print("== passage_processing ==")
    for st, rs, n in c.execute(text(f"SELECT state, reason, count(*) FROM passage_processing WHERE {SCOPE} GROUP BY 1,2 ORDER BY 1,2"), P): print(f"  {st:16s} {rs:34s} {n:5d}")
    print("== other rows ==")
    for t, w in [("validation_failure", "AND stage='extraction'"), ("alias_evidence", "AND extraction_run_id IS NOT NULL"),
                 ("entity_mention", ""), ("assertion", ""), ("materialization_run", "")]:
        print(f"  {t:22s} {c.execute(text(f'SELECT count(*) FROM {t} WHERE {SCOPE} {w}'), P).scalar():5d}")
with engine.connect() as c: snapshot(c)

## 2. Reset
Leave `DRY_RUN = True` first: it does everything, prints the result, then rolls back. Set to `False` to commit.

In [ ]:
DRY_RUN = True

class _Rollback(Exception): pass

def reset():
    try:
        with engine.begin() as c:
            blocked = {t: c.execute(text(f"SELECT count(*) FROM {t} WHERE {SCOPE}"), P).scalar() for t in ("entity_mention", "assertion", "materialization_run")}
            if any(blocked.values()):
                raise RuntimeError(f"extractions already materialized {blocked}; clear/rebuild the graph first")
            r = c.execute(text(f"""UPDATE passage_processing
                  SET state='selected', reason='extraction_reset', active_extraction_run_id=NULL,
                      validated_extraction_run_id=NULL, updated_at=now()
                  WHERE {SCOPE} AND active_jev_selection_id IS NOT NULL AND state IN ('selected','extracted','failed')"""), P)
            print("passages reset to 'selected':", r.rowcount)
            for label, sql in [
                ("alias_evidence", f"DELETE FROM alias_evidence WHERE {SCOPE} AND extraction_run_id IS NOT NULL"),
                ("validation_failure", f"DELETE FROM validation_failure WHERE {SCOPE} AND stage='extraction'"),
                ("llm_extraction_run", f"DELETE FROM llm_extraction_run WHERE {SCOPE}"),
            ]:
                print(f"deleted {label}:", c.execute(text(sql), P).rowcount)
            print("\n--- state after reset ---"); snapshot(c)
            if DRY_RUN: raise _Rollback
    except _Rollback:
        print("\nDRY_RUN: rolled back, nothing changed.")
reset()

## 3. Re-run
From a shell: run the extract stage (see `README`/`bin/runner`), or in a notebook cell:
```python
from potency_kg.workflow_stages import extract_stage
extract_stage(config_path)   # config_path = root / "configs/casgevy-potency.json"
```
Then open `04_monitor_extraction.ipynb`; failed runs now carry `error_detail` (real exception) and `raw_response` (model output before validation).